## 5. GRU Baseline

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [ ]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

MODEL_NAME = 'GRU'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gru'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gru_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gru_val_stability_summary.json'

In [ ]:
stats = load_cgm_stats()
train_loader = load_tensors('train')
val_loader = load_tensors('val')
test_loader = load_tensors('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = GRUPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gru' /f'gru_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gru_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f"\n{MODEL_NAME} | seed={seed} | stopped at epoch {n_epochs}/50")

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


Validation results — GRU | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]


GRU | seed=7 | stopped at epoch 42/50
 15 min | RMSE 11.01 | MAE 7.45 | gRMSE 12.19 | Zone A 96.86% | A+B 99.65%
 30 min | RMSE 17.50 | MAE 11.75 | gRMSE 19.96 | Zone A 90.11% | A+B 99.31%
 60 min | RMSE 24.89 | MAE 16.77 | gRMSE 29.67 | Zone A 81.59% | A+B 98.67%
 90 min | RMSE 29.10 | MAE 19.74 | gRMSE 35.38 | Zone A 76.50% | A+B 98.16%
120 min | RMSE 31.61 | MAE 21.69 | gRMSE 38.79 | Zone A 72.84% | A+B 97.89%

--------------------------------------------------------------------------------
Validation results — GRU | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]


GRU | seed=14 | stopped at epoch 35/50
 15 min | RMSE 10.97 | MAE 7.34 | gRMSE 11.96 | Zone A 96.79% | A+B 99.68%
 30 min | RMSE 17.45 | MAE 11.60 | gRMSE 19.65 | Zone A 89.95% | A+B 99.35%
 60 min | RMSE 24.81 | MAE 16.57 | gRMSE 29.26 | Zone A 81.84% | A+B 98.71%
 90 min | RMSE 29.03 | MAE 19.63 | gRMSE 34.92 | Zone A 76.93% | A+B 98.21%
120 min | RMSE 31.63 | MAE 21.69 | gRMSE 38.46 | Zone A 73.23% | A+B 97.94%

--------------------------------------------------------------------------------
Validation results — GRU | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


...


GRU | seed=56 | stopped at epoch 15/50
 15 min | RMSE 11.19 | MAE 7.61 | gRMSE 12.14 | Zone A 96.32% | A+B 99.56%
 30 min | RMSE 17.71 | MAE 11.83 | gRMSE 19.95 | Zone A 89.44% | A+B 99.28%
 60 min | RMSE 25.07 | MAE 16.54 | gRMSE 29.72 | Zone A 81.68% | A+B 98.70%
 90 min | RMSE 29.35 | MAE 19.59 | gRMSE 35.49 | Zone A 76.81% | A+B 98.13%
120 min | RMSE 31.97 | MAE 21.72 | gRMSE 39.05 | Zone A 73.45% | A+B 97.79%

--------------------------------------------------------------------------------
Validation results — GRU | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]


GRU | seed=63 | stopped at epoch 48/50
 15 min | RMSE 10.88 | MAE 7.37 | gRMSE 11.73 | Zone A 96.77% | A+B 99.68%
 30 min | RMSE 17.45 | MAE 11.63 | gRMSE 19.48 | Zone A 89.79% | A+B 99.33%
 60 min | RMSE 24.93 | MAE 16.77 | gRMSE 29.25 | Zone A 81.29% | A+B 98.69%
 90 min | RMSE 29.27 | MAE 19.95 | gRMSE 35.09 | Zone A 75.86% | A+B 98.20%
120 min | RMSE 31.87 | MAE 22.08 | gRMSE 38.56 | Zone A 72.32% | A+B 98.00%

--------------------------------------------------------------------------------
Validation results — GRU | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]


GRU | seed=70 | stopped at epoch 44/50
 15 min | RMSE 10.95 | MAE 7.39 | gRMSE 11.81 | Zone A 96.77% | A+B 99.68%
 30 min | RMSE 17.49 | MAE 11.64 | gRMSE 19.65 | Zone A 89.88% | A+B 99.34%
 60 min | RMSE 24.91 | MAE 16.61 | gRMSE 29.39 | Zone A 81.71% | A+B 98.70%
 90 min | RMSE 29.21 | MAE 19.77 | gRMSE 35.13 | Zone A 76.50% | A+B 98.18%
120 min | RMSE 31.79 | MAE 21.96 | gRMSE 38.60 | Zone A 72.78% | A+B 97.92%

--------------------------------------------------------------------------------


In [ ]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [ ]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [ ]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== GRU validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.96 +/- 0.09 (2.38) mg/dL   MAE = 7.40 +/- 0.09 (1.60)   gRMSE = 11.92 +/- 0.18 (3.26)   Zone A = 96.71 +/- 0.16 (3.08)%   Zone A+B = 99.66 +/- 0.04 (0.68)%
 30-min:  RMSE = 17.48 +/- 0.10 (3.80) mg/dL   MAE = 11.65 +/- 0.09 (2.43)   gRMSE = 19.69 +/- 0.17 (5.68)   Zone A = 89.91 +/- 0.22 (5.56)%   Zone A+B = 99.33 +/- 0.03 (0.97)%
 60-min:  RMSE = 24.89 +/- 0.08 (7.35) mg/dL   MAE = 16.62 +/- 0.13 (4.67)   gRMSE = 29.43 +/- 0.16 (11.02)   Zone A = 81.69 +/- 0.23 (5.35)%   Zone A+B = 98.69 +/- 0.02 (1.43)%
 90-min:  RMSE = 29.16 +/- 0.10 (10.25) mg/dL   MAE = 19.73 +/- 0.16 (6.69)   gRMSE = 35.19 +/- 0.17 (15.14)   Zone A = 76.59 +/- 0.35 (7.10)%   Zone A+B = 98.18 +/- 0.03 (2.00)%
120-min:  RMSE = 31.76 +/- 0.11 (12.02) mg/dL   MAE = 21.84 +/- 0.15 (8.03)   gRMSE = 38.69 +/- 0.19 (17.87)   Zone A = 72.88 +/- 0.35 (8.15)%   Zone A+B = 97.89 +/- 0.07 (2.47)%


In [ ]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/gru/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/gru/gru_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/gru/gru_stability_summary.json
